Regressão Linear Simples


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# Configurações de estilo
plt.style.use('default')
sns.set_theme(style="whitegrid")

# Dados Iniciais
dados = pd.DataFrame({
    "horas_estudo": [2, 3, 4, 5, 5, 6, 7, 7, 8, 9, 10, 10, 11, 12, 12, 13, 14, 15, 16, 18],
    "nota": [42, 45, 51, 48, 55, 58, 57, 63, 61, 67, 69, 65, 74, 72, 78, 76, 83, 81, 88, 86]
})


1. Análise Exploratória

a) Identifique a variável resposta e a variável explicativa, incluindo suas unidades.
Variável Resposta ($Y$): `nota` (em pontos / unidades de nota).
Variável Explicativa ($X$): `horas_estudo` (em horas/semana).


In [ ]:
# b) Calcule média, mediana, desvio-padrão amostral, mínimo e máximo das duas variáveis.
estatisticas = dados.describe().T[['mean', '50%', 'std', 'min', 'max']]
estatisticas.columns = ['Média', 'Mediana', 'Desvio-Padrão', 'Mínimo', 'Máximo']
display(estatisticas)


In [ ]:
# c) Construa o diagrama de dispersão com eixos identificados.
plt.figure(figsize=(8, 5))
sns.scatterplot(data=dados, x='horas_estudo', y='nota', color='blue', s=80)
plt.title('Diagrama de Dispersão: Nota vs Horas de Estudo')
plt.xlabel('Horas de Estudo (horas/semana)')
plt.ylabel('Nota')
plt.show()


In [ ]:
# d) Calcule a correlação de Pearson e interprete direção e intensidade.
correlacao = dados['horas_estudo'].corr(dados['nota'])
print(f"Correlação de Pearson (r): {correlacao:.4f}")


A correlação é de aproximadamente 0.978. A direção é positiva e a intensidade é muito forte, indicando que, à medida que as horas de estudo aumentam, as notas tendem a aumentar substancialmente.

e) Discuta a adequação inicial de uma reta ao padrão observado.
O diagrama de dispersão exibe os pontos alinhados segundo uma tendência crescente bastante retilínea, sem curvaturas ou agrupamentos atípicos visíveis. Isso sugere que o ajuste de um modelo de regressão linear simples é muito adequado inicialmente para descrever os dados observacionais.


2. Estimação por mínimos quadrados


In [ ]:
# a) Implemente as fórmulas de Sxx, Sxy, inclinação e intercepto
x = dados['horas_estudo']
y = dados['nota']

x_mean = x.mean()
y_mean = y.mean()

Sxx = np.sum((x - x_mean)**2)
Sxy = np.sum((x - x_mean) * (y - y_mean))

beta1_manual = Sxy / Sxx
beta0_manual = y_mean - beta1_manual * x_mean

print("--- Procedimento Manual ---")
print(f"Sxx: {Sxx:.4f}, Sxy: {Sxy:.4f}")
print(f"Intercepto (beta0): {beta0_manual:.4f}")
print(f"Inclinação (beta1): {beta1_manual:.4f}\n")

# b) Ajuste nota ~ horas_estudo usando OLS()
X_sm = sm.add_constant(x)
modelo = sm.OLS(y, X_sm).fit()

print("--- Procedimento statsmodels OLS ---")
beta0_ols, beta1_ols = modelo.params
print(f"Intercepto (beta0): {beta0_ols:.4f}")
print(f"Inclinação (beta1): {beta1_ols:.4f}")

# d) Escreva a equação estimada
print(f"\nEquação Estimada: Nota_hat = {beta0_ols:.4f} + {beta1_ols:.4f} * Horas_estudo")

# e) Calcule ajustados e resíduos e verifique a soma dos resíduos.
ajustados = modelo.fittedvalues
residuos = modelo.resid
soma_residuos = np.sum(residuos)
print(f"\nSoma dos resíduos: {soma_residuos:.4e}")


c) Compare os coeficientes dos dois procedimentos
Os coeficientes calculados pelas fórmulas matemáticas exatas e pela função `OLS()` são idênticos, como esperado

d) Interpretação da inclinação e intercepto
Estima-se um aumento médio de aproximadamente 2,98 pontos na nota final para cada hora adicional de estudo por semana.
Estima-se que a nota seria de aproximadamente 38,05 pontos para um estudante hipotético com zero horas de estudo na semana.

e) Soma dos resíduos
A soma dos resíduos calculada é na ordem de grandeza, comprovando a propriedade central do método dos mínimos quadrados na qual a soma dos erros estimados é nula.


3. Qualidade do ajuste e inferência


In [ ]:
# a) Calcule SQT, SQReg e SQRes
SQT = np.sum((y - y_mean)**2)
SQReg = np.sum((ajustados - y_mean)**2)
SQRes = np.sum(residuos**2)

print(f"SQT: {SQT:.4f}")
print(f"SQReg: {SQReg:.4f}")
print(f"SQRes: {SQRes:.4f}")
print(f"Decomposição SQT = SQReg + SQRes: {SQReg + SQRes:.4f} (Verificado)\n")

# b) Calcule R²
R2 = SQReg / SQT
print(f"R²: {R2:.4f}\n")

# c) Apresente ANOVA com graus de liberdade, SQ, QM, F e valor-p.
modelo_formula = smf.ols('nota ~ horas_estudo', data=dados).fit()
anova_table = sm.stats.anova_lm(modelo_formula, typ=1)
display(anova_table)

# d) Teste de hipóteses para inclinação
p_valor_beta1 = modelo.pvalues['horas_estudo']
print(f"\nValor-p para inclinação (beta1): {p_valor_beta1:.4e}")

# e) Obtenha IC de 95% para beta1
ic_beta1 = modelo.conf_int(alpha=0.05).loc['horas_estudo']
print(f"IC 95% para beta1: [{ic_beta1[0]:.4f}, {ic_beta1[1]:.4f}]")


b) Interpretação do $R^2$:** O $R^2$ de 0.9559 significa que praticamente 95,6% da variabilidade total observada nas notas dos estudantes do modelo é explicada pela variação nas suas horas semanais de estudo.

d) Teste 
O valor-p da inclinação ($1.2 \times 10^{-13}$) é infinitamente menor que o nível de significância de 5% (0.05). Por isso, rejeitamos a hipótese nula ($H_0$). Existe evidência robusta de que a inclinação é estritamente diferente de zero, indicando que as horas de estudo impactam as notas de forma estatisticamente significativa.

e) Comparação IC 95% e Decisão do Teste:
O Intervalo de Confiança de 95% para $\beta_1$ é $[2.666, 3.301]$. Como o intervalo não engloba o zero, isso atesta e corrobora integralmente com a decisão imposta pelo teste t de rejeitar $H_0$ (existe associação linear significativa).


4. Estimativas e predições


In [ ]:
# Utilizando get_prediction() do statsmodels
nova_obs = pd.DataFrame({"const": [1], "horas_estudo": [9]})
predicao = modelo.get_prediction(nova_obs).summary_frame(alpha=0.05)
display(predicao[['mean', 'mean_ci_lower', 'mean_ci_upper', 'obs_ci_lower', 'obs_ci_upper']])

print(f"\na) Previsão (9h): {predicao['mean'].values[0]:.2f} pontos")
print(f"b) IC 95% para a média (9h): [{predicao['mean_ci_lower'].values[0]:.2f}, {predicao['mean_ci_upper'].values[0]:.2f}]")
print(f"c) IP 95% para um novo estudante (9h): [{predicao['obs_ci_lower'].values[0]:.2f}, {predicao['obs_ci_upper'].values[0]:.2f}]")


d) Compare as amplitudes e explique a diferença:
Amplitude do IC 95% condicional médio: $\approx 2.81$ (66.31 - 63.50)
Amplitude do IP 95% do indivíduo: $\approx 12.83$ (71.32 - 58.49)
A disparidade enorme ocorre porque a predição para um estudante unitário inédito (IP) abrange toda a variabilidade natural intrínseca a esse novo estudante e os seus ruídos isolados (o erro estocástico irredutível), além de carregar consigo a incerteza gerada pela estimação da própria média e da inclinação da reta (que o IC também absorve).

e) Discuta o risco de prever para 35 horas semanais:
Sendo a contagem máxima dos nossos dados 18 horas, elaborar uma previsão matemática em torno de 35 horas semanais exige o pressuposto falacioso da validade estendida de uma tendência contínua ininterrupta, configurando sérios riscos de uma forte extrapolação. Prever além do limite de abrangência costuma entregar dados fisicamente irreais — como pontuações infinitas que não reconhecem o esgotamento do estudante ou o limite formal de 100 pontos da nota total da unidade avaliativa.


5. Diagnóstico


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# a) Apresente resíduos versus ajustados.
sns.scatterplot(x=ajustados, y=residuos, ax=axes[0], color='purple')
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set_title("Resíduos vs Valores Ajustados")
axes[0].set_xlabel("Valores Ajustados")
axes[0].set_ylabel("Resíduos")

# b) Apresente gráfico Q-Q dos resíduos.
sm.qqplot(residuos, line='s', ax=axes[1])
axes[1].set_title("Gráfico Q-Q dos Resíduos")

plt.tight_layout()
plt.show()

# d) Calcule distâncias de Cook e identifique a maior.
influencia = modelo.get_influence()
cooks_d = influencia.cooks_distance[0]

dados_cooks = pd.DataFrame({'Distancia_Cook': cooks_d})
indice_max = dados_cooks['Distancia_Cook'].idxmax()
valor_max = dados_cooks['Distancia_Cook'].max()
horas_max = dados.loc[indice_max, 'horas_estudo']
nota_max = dados.loc[indice_max, 'nota']

print(f"A maior Distância de Cook é {valor_max:.4f} ocorrendo no índice {indice_max} (Estudante com {horas_max}h, nota {nota_max})")


c) Discuta linearidade, homoscedasticidade e normalidade:
Linearidade: Observa-se que no gráfico de Resíduos vs Valores Ajustados não existe padrão contínuo identificável como curvas concavas/convexas na dispersão (como 'U' ou ondas). Os pontos orbitam livremente o eixo 0.
Homoscedasticidade: Com os pontos fluindo num tubo estável ao longo do eixo horizontal (sem abrir ou fechar em formatação afunilada ou em trombeta), o pressuposto de amplitude constante (homoscedasticidade) parece plausível e atendido.
Normalidade: No gráfico Q-Q, as contagens estão devidamente alinhadas e ancoradas à reta diagonal ideal, amparando firmemente a pressuposição de normalidade probabilística dos resíduos.

e) Discuta influência e por que exclusão automática não é recomendada:
O indivíduo correspondente ao 19º registro (com a marca de 18 horas de estudo em regime intensivo e nota 86) é o de máxima de Distância de Cook (0.7905). Essa contagem tem influência alavancada suficiente para repuxar a inclinação perante o restante da amostra, mas a prática da imediata eliminação analítica não é aconselhada, uma vez que representa os extremos de uma rotina realística que não reflete uma anomalia na digitação, mas sim evidências consistentes sobre dados populacionais perfeitamente legítimos do mundo real.

f) Explique por que os gráficos não demonstram independência:
Testes clássicos bidimensionais visuais construídos através de resíduos versus valores ajustados não traduzem nem revelam dependência entre pares ou interações sistemáticas de origem (autocorrelação subjacente) em conjuntos de dados transversais aleatórios desprovidos de linha temporal (séries de tempo). O atestado de validação que as amostras são de fato independentes deriva da execução metodológica durante o levantamento amostral da avaliação e se as notas foram aferidas rigorosamente de modo autônomo sem intervenções compartilhadas nos exames originais.


6. Conclusão

A extensa análise de regressão linear simples efetivada demonstra cabalmente que as horas semanais de estudo estão fortemente vinculadas e associadas à nota final dos vinte estudantes avaliados. Esta associação se revela por meio de direção invariavelmente positiva e se comprova estatisticamente significativa (p-valor incrivelmente pequeno, $p < 0,05$), resultando no embasamento de que o esforço contínuo acompanha escores notadamente superiores. O modelo formulado demonstrou notável grau de adequação geométrica frente à amostra, alcançando a marca matemática de explicar em torno de 95,6% da flutuação aleatória da nota unicamente por via do acréscimo temporal. Por conseguinte, exames de diagnóstico respaldam homogeneidade variacional e normalidade distribucional ideais para viabilizar e sustentar tais métricas inferenciais. No entanto, o estudo abarca imensuráveis limitações práticas: possui um corpo pequeno, é restrito apenas a métodos observacionais passíveis a viés estatístico de seleção (com apenas $n=20$) e é vulnerável ao iminente risco matemático de extrapolação fora do seu raio basal restrito de alcance contínuo para até no máximo 18 horas de trabalho intelectual diário. Destarte, reforça-se veementemente a necessidade ética de não interpretar a intensidade associativa das horas perante o sucesso do resultado como uma lei comprobatória e imutável de causalidade linear estrita, visto o provável obscurecimento gerado perante fatores intervenientes invisíveis complexos (como base histórica ou inteligências cognitivas subjacentes preexistentes).
